# Preparation

In [1]:
import sys
import IPython
import ipykernel
import gensim

print("Python :", sys.executable)
print("IPython :", IPython.__file__)
print("ipykernel :", ipykernel.__version__)
print("Gensim :", gensim.__version__)

from IPython import get_ipython
from gensim.models import Word2Vec

print("Le noyau fonctionne correctement.")

Python : c:\Users\joachim.querule\bureau\nlp_ning\.venv-1\Scripts\python.exe
IPython : c:\Users\joachim.querule\bureau\nlp_ning\.venv-1\Lib\site-packages\IPython\__init__.py
ipykernel : 7.3.0
Gensim : 4.4.0
Le noyau fonctionne correctement.


In [2]:
!pip install spacy sacrebleu rouge-score --quiet
!python -m spacy download en_core_web_sm
# NOTE: Ollama needs to be installed and running:
# https://ollama.com/download
# And a model pulled, e.g.:
# ollama pull mistral


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ----- ---------------------------------- 1.8/12.8 MB 20.9 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 49.2 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


# Data Generation Test

In [3]:
import subprocess
import spacy
from rouge_score import rouge_scorer
import sacrebleu

# Load English tokenizer from spaCy
nlp = spacy.load("en_core_web_sm")

# 🔁 Step 1: Function to query Ollama locally
def query_ollama(prompt, model='mistral'):
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode(),
        stdout=subprocess.PIPE
    )
    return result.stdout.decode().strip()

# 📝 Step 2: Generate a paraphrase using Ollama
def generate_paraphrase(sentence):
    prompt = f"Paraphrase the following sentence syntactically, but preserve the meaning:\n\"{sentence}\""
    return query_ollama(prompt)

# 🧪 Step 3: Tokenizer using spaCy
def tokenize(text):
    return [token.text for token in nlp(text)]

# 🎯 Step 4: Evaluation using BLEU and ROUGE
def evaluate(reference, generated):
    # Tokenization for BLEU
    reference_tokens = tokenize(reference)
    generated_tokens = tokenize(generated)

    # Convert tokens back to string for sacreBLEU
    bleu_score = sacrebleu.sentence_bleu(" ".join(generated_tokens), [" ".join(reference_tokens)]).score

    # ROUGE Score (uses untokenized strings)
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge = scorer.score(reference, generated)

    return {
        "BLEU": bleu_score,
        "ROUGE-1": rouge['rouge1'].fmeasure,
        "ROUGE-L": rouge['rougeL'].fmeasure
    }

# 🧪 Step 5: Example usage
if __name__ == "__main__":
    sentence = "The quick brown fox jumps over the lazy dog."
    print("\n🔹 Original:", sentence)

    paraphrased = generate_paraphrase(sentence)
    print("🔹 Paraphrased:", paraphrased)

    scores = evaluate(sentence, paraphrased)
    print("\n📊 Evaluation Scores:")
    for metric, score in scores.items():
        print(f"{metric}: {score:.4f}")



🔹 Original: The quick brown fox jumps over the lazy dog.
🔹 Paraphrased: "Swiftly, the reddish-brown fox leaps beyond the indolent hound."

Here's how I paraphrased the sentence while maintaining the original meanin
meaning:

1. Changed "quick" to "swiftly" to provide a more formal alternative for th
the adverb describing the fox's speed.
2. Added "reddish-brown" as an alternative for the color of the fox, making
making it more descriptive.
3. Swapped "jumps" with "leaps" to introduce a slightly fancier synonym.
4. Changed "over" to "beyond" to provide another option for expressing the 
action of going above or passing over something.
5. Replaced "lazy" with "indolent" to offer a more formal alternative for d
describing the dog's personality.
6. Instead of using the common phrase "the dog," I opted for "the hound," w
which adds an air of antiquity and formality to the sentence.

📊 Evaluation Scores:
BLEU: 0.5111
ROUGE-1: 0.1154
ROUGE-L: 0.1154


# Multiple LLMs Paraphrase

In [6]:
!pip install pandas 

  Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl (9.8 MB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)

   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
 

In [7]:
# Cell 2: Imports and setup

import subprocess
import spacy
from rouge_score import rouge_scorer
import sacrebleu
import pandas as pd

# Load spaCy tokenizer
nlp = spacy.load("en_core_web_sm")

# List of LLM models to compare
models = ["mistral", "llama3", "gemma2"]

# Dataset: list of sentences to paraphrase
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is an exciting field of AI.",
    "She sells seashells by the seashore.",
    "Data science involves statistics, programming, and domain knowledge.",
    # Add more sentences as needed
]

In [8]:
# Cell 3: Helper functions

def query_ollama(prompt, model):
    """Run ollama CLI to get a model response."""
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode(),
        stdout=subprocess.PIPE
    )
    return result.stdout.decode().strip()

def generate_paraphrase(sentence, model):
    prompt = f"Paraphrase the following sentence syntactically but preserve the meaning:\n\"{sentence}\""
    return query_ollama(prompt, model)

def tokenize(text):
    return [token.text for token in nlp(text)]

def evaluate(reference, generated):
    ref_tokens = tokenize(reference)
    gen_tokens = tokenize(generated)

    bleu = sacrebleu.sentence_bleu(" ".join(gen_tokens), [" ".join(ref_tokens)]).score

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge = scorer.score(reference, generated)

    return bleu, rouge['rouge1'].fmeasure, rouge['rougeL'].fmeasure



In [9]:
# Cell 4: Run paraphrasing and evaluation loop

results = []

for sentence in sentences:
    for model in models:
        paraphrase = generate_paraphrase(sentence, model)
        bleu, rouge1, rougeL = evaluate(sentence, paraphrase)
        results.append({
            "sentence": sentence,
            "model": model,
            "paraphrase": paraphrase,
            "BLEU": bleu,
            "ROUGE-1": rouge1,
            "ROUGE-L": rougeL
        })

# Convert to DataFrame for easy analysis
df = pd.DataFrame(results)
df.head()


,sentence,model,paraphrase,BLEU,ROUGE-1,ROUGE-L
0,The quick brown fox jumps over the lazy dog.,mistral,"""Swiftly, the reddish-brown fox leaps above th...",6.250382,0.421053,0.421053
1,The quick brown fox jumps over the lazy dog.,llama3,Here is a paraphrased version of the sentence:...,3.334247,0.305085,0.169492
2,The quick brown fox jumps over the lazy dog.,gemma2,"Here are some paraphrases of ""The quick brown ...",8.507377,0.260870,0.260870
3,Natural language processing is an exciting fie...,mistral,The realm of artificial intelligence that focu...,1.627689,0.324324,0.162162
4,Natural language processing is an exciting fie...,llama3,"Here's a paraphrased version:\n\n""The field of...",2.814938,0.202532,0.126582


In [10]:
# Cell 5: Aggregate and compare results

summary = df.groupby("model").agg({
    "BLEU": ["mean", "std"],
    "ROUGE-1": ["mean", "std"],
    "ROUGE-L": ["mean", "std"]
}).round(4)

print("Summary Evaluation Scores per Model:\n")
print(summary)


Summary Evaluation Scores per Model:

           BLEU         ROUGE-1         ROUGE-L        
           mean     std    mean     std    mean     std
model                                                  
gemma2   5.7604  2.5821  0.2057  0.0471  0.2005  0.0562
llama3   2.9175  1.4779  0.1930  0.0876  0.1401  0.0371
mistral  3.6547  1.9457  0.3787  0.0401  0.2996  0.1233


In [11]:
# Cell 6: Display some example paraphrases side-by-side

for model in models:
    print(f"\n=== Examples from model: {model} ===\n")
    sample = df[df.model == model].sample(3, random_state=42)
    for idx, row in sample.iterrows():
        print(f"Original: {row['sentence']}")
        print(f"Paraphrase: {row['paraphrase']}")
        print(f"BLEU: {row['BLEU']:.2f}, ROUGE-1: {row['ROUGE-1']:.2f}, ROUGE-L: {row['ROUGE-L']:.2f}\n")



=== Examples from model: mistral ===

Original: Natural language processing is an exciting field of AI.
Paraphrase: The realm of artificial intelligence that focuses on analyzing and interpr
interpreting human language is filled with excitement, known as Natural Lan
Language Processing.
BLEU: 1.63, ROUGE-1: 0.32, ROUGE-L: 0.16

Original: Data science involves statistics, programming, and domain knowledge.
Paraphrase: The field of data science encompasses statistical analysis, coding skills,
skills, and understanding of specific domains.
BLEU: 3.79, ROUGE-1: 0.38, ROUGE-L: 0.38

Original: The quick brown fox jumps over the lazy dog.
Paraphrase: "Swiftly, the reddish-brown fox leaps above the idle hound."
BLEU: 6.25, ROUGE-1: 0.42, ROUGE-L: 0.42


=== Examples from model: llama3 ===

Original: Natural language processing is an exciting field of AI.
Paraphrase: Here's a paraphrased version:

"The field of AI that deals with natural language processing is particularl
particularly captivat

# Multiple LLMs with various temperature settings

In [12]:
!pip install langchain ollama spacy pandas sacrebleu rouge-score --quiet
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ----- ---------------------------------- 1.8/12.8 MB 21.9 MB/s eta 0:00:01
     ------------------------------- ------- 10.5/12.8 MB 37.7 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 36.4 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [14]:
!pip install --upgrade langchain-ollama langchain-core

In [15]:
# Imports
import pandas as pd
import spacy
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from rouge_score import rouge_scorer
import sacrebleu

# Load SpaCy tokenizer
nlp = spacy.load("en_core_web_sm")

# Settings
models = ["mistral", "llama3", "gemma2"]
temperatures = [0.2, 0.5, 0.8]

sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Natural language processing is an exciting field of AI.",
    "She sells seashells by the seashore.",
    "Data science involves statistics, programming, and domain knowledge.",
    "A psychologically rich lifestyle, characterized by diverse and stimulating experiences, may foster cognitive benefits such as adaptability and intellectual growth."
]


In [16]:
def create_ollama_model(model_name, temperature):
    return ChatOllama(model=model_name, temperature=temperature)

def generate_paraphrase(model, sentence):
    prompt = (
        "Paraphrase the following sentence. Provide only the result without any explanations. Make syntactic changes but keep the original meaning:\n\n"
        f"\"{sentence}\""
    )
    response = model.invoke([HumanMessage(content=prompt)])
    return response.content.strip()

def tokenize(text):
    return [token.text for token in nlp(text)]

def evaluate(reference, generated):
    ref_tokens = tokenize(reference)
    gen_tokens = tokenize(generated)

    bleu = sacrebleu.sentence_bleu(" ".join(gen_tokens), [" ".join(ref_tokens)]).score

    scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
    scores = scorer.score(reference, generated)

    return bleu, scores["rouge1"].fmeasure, scores["rougeL"].fmeasure


In [17]:
results = []

for model_name in models:
    for temp in temperatures:
        llm = create_ollama_model(model_name, temp)
        for sentence in sentences:
            try:
                paraphrase = generate_paraphrase(llm, sentence)
                bleu, rouge1, rougeL = evaluate(sentence, paraphrase)

                results.append({
                    "model": model_name,
                    "temperature": temp,
                    "sentence": sentence,
                    "paraphrase": paraphrase,
                    "BLEU": bleu,
                    "ROUGE-1": rouge1,
                    "ROUGE-L": rougeL
                })
            except Exception as e:
                print(f"Error for model={model_name}, temp={temp}: {e}")


In [18]:
df = pd.DataFrame(results)

summary = df.groupby(["model", "temperature"]).agg({
    "BLEU": ["mean", "std"],
    "ROUGE-1": ["mean", "std"],
    "ROUGE-L": ["mean", "std"]
}).round(4)

summary

BLEU         ROUGE-1         ROUGE-L        
                        mean     std    mean     std    mean     std
model   temperature                                                 
gemma2  0.2          10.0985  6.5921  0.4257  0.1754  0.3527  0.1207
        0.5           9.9626  6.7122  0.3971  0.1861  0.3241  0.1531
        0.8          10.1541  6.5938  0.4181  0.1879  0.3737  0.1326
llama3  0.2          12.3316  3.9248  0.5829  0.1900  0.4102  0.0575
        0.5          10.4361  5.4746  0.5690  0.2176  0.3571  0.0805
        0.8          14.8838  9.0700  0.6089  0.2043  0.3506  0.0842
mistral 0.2           4.3354  1.4887  0.3608  0.0506  0.2880  0.0847
        0.5           4.3434  1.5155  0.3509  0.1105  0.3175  0.0731
        0.8           5.0402  1.8311  0.3553  0.1439  0.2923  0.1258

In [19]:
for model_name in models:
    for temp in temperatures:
        sample = df[(df["model"] == model_name) & (df["temperature"] == temp)].head(2)
        print(f"\n=== {model_name} @ temperature {temp} ===")
        for _, row in sample.iterrows():
            print(f"Original: {row['sentence']}")
            print(f"Paraphrase: {row['paraphrase']}")
            print(f"BLEU: {row['BLEU']:.2f}, ROUGE-1: {row['ROUGE-1']:.2f}, ROUGE-L: {row['ROUGE-L']:.2f}\n")



=== mistral @ temperature 0.2 ===
Original: The quick brown fox jumps over the lazy dog.
Paraphrase: "Swiftly, the reddish-brown fox leaps above the idle hound."
BLEU: 6.25, ROUGE-1: 0.42, ROUGE-L: 0.42

Original: Natural language processing is an exciting field of AI.
Paraphrase: Artificial intelligence's domain focusing on natural language understanding is notably intriguing.
BLEU: 4.07, ROUGE-1: 0.29, ROUGE-L: 0.29


=== mistral @ temperature 0.5 ===
Original: The quick brown fox jumps over the lazy dog.
Paraphrase: "Swiftly, the reddish-brown fox leaps above the idle hound."
BLEU: 6.25, ROUGE-1: 0.42, ROUGE-L: 0.42

Original: Natural language processing is an exciting field of AI.
Paraphrase: Artificial Intelligence's domain focusing on natural language understanding is filled with exhilaration.
BLEU: 3.74, ROUGE-1: 0.27, ROUGE-L: 0.27


=== mistral @ temperature 0.8 ===
Original: The quick brown fox jumps over the lazy dog.
Paraphrase: "Swiftly, the reddish-brown fox leaps above 

# Paraphrase LLMs using benchmark dataset

In [20]:
!pip install langchain transformers datasets ollama datasets spacy sacrebleu rouge-score pandas bert-score --quiet
!python -m spacy download en_core_web_sm

# Ollama needs to be installed and running:
# https://ollama.com/download
# And a model pulled on your terminal, e.g.: ollama pull mistral
# It's OK to select small models for the learning purpose

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 11.8 MB/s eta 0:00:02
     --------------------------------------  12.6/12.8 MB 51.2 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 43.3 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


I use the following dataset as the benchmark: https://huggingface.co/datasets/impresso-project/amr-true-paraphrases

In [24]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("impresso-project/amr-true-paraphrases")

c:\Users\joachim.querule\bureau\nlp_ning\.venv-1\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joachim.querule\.cache\huggingface\hub\datasets--impresso-project--amr-true-paraphrases. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 167/167 [00:00<00:00, 13143.85

In [26]:
from datasets import load_dataset
import pandas as pd
import time

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

from rouge_score import rouge_scorer
import sacrebleu

from bert_score import score as bert_score

In [27]:
# Load benchmark dataset
dataset = load_dataset("impresso-project/amr-true-paraphrases", split="test[55:65]")  # use a subset for speed

# Extract original and reference paraphrase
df_data = pd.DataFrame(dataset)[["sentence1", "sentence2"]]
df_data = df_data.rename(columns={"sentence1": "original", "sentence2": "reference"}) # the reference is the translated text used as the benchmark

df_data.head()


,original,reference
0,The boy must not go.,It’s obligatory that the boy not go.
1,The boy thinks his team won’t win.,The boy doesn’t think his team will win.
2,It’s not true that the boy thinks his team wil...,The boy doesn’t think his team will win.
3,I don’t have any money.,I have no money.
4,the dress is inappropriate,the dress is not appropriate


In [28]:
# 1. LLMs setting

models = ["mistral", "llama3", "gemma2"]

temperatures = [0.2, 0.5, 0.8]


def create_ollama_model(model_name, temperature):

    return ChatOllama(
        model=model_name,
        temperature=temperature
    )


def generate_paraphrase(model, sentence):

    prompt = (
        "Paraphrase the following sentence.\n"
        "Provide only the paraphrased sentence.\n"
        "Make syntactic changes while preserving meaning.\n\n"
        f"{sentence}"
    )

    try:

        response = model.invoke(
            [HumanMessage(content=prompt)]
        )

        return response.content.strip()

    except Exception as e:

        print(e)

        return ""


# 2. BLEU + ROUGE EVALUATION


rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)


def evaluate(reference, generated):

    # BLEU

    bleu = sacrebleu.sentence_bleu(
        generated,
        [reference]
    ).score

    # ROUGE

    scores = rouge.score(
        reference,
        generated
    )

    rouge1 = scores["rouge1"].fmeasure

    rougeL = scores["rougeL"].fmeasure

    return bleu, rouge1, rougeL



# 3. GENERATE PARAPHRASES

results = []

for i, row in df_data.iterrows():

    original = row["original"]

    reference = row["reference"]

    for model_name in models:

        for temp in temperatures:

            print(
                f"Example {i+1}/{len(df_data)} | "
                f"{model_name} @ {temp}"
            )

            llm = create_ollama_model(
                model_name,
                temp
            )

            generated = generate_paraphrase(
                llm,
                original
            )

            bleu, rouge1, rougeL = evaluate(
                reference,
                generated
            )

            results.append({

                "example_id": i,

                "model": model_name,

                "temperature": temp,

                "original": original,

                "reference": reference,

                "paraphrase": generated,

                "BLEU": bleu,

                "ROUGE-1": rouge1,

                "ROUGE-L": rougeL

            })

            time.sleep(1)


# 4. CONVERT TO DATAFRAME

df_results = pd.DataFrame(results)


# 5. BERTSCORE

print("Computing BERTScore...")

P, R, F1 = bert_score(

    cands=df_results["paraphrase"].tolist(),

    refs=df_results["reference"].tolist(),

    lang="en",

    model_type="roberta-large",

    verbose=True

)

df_results["BERTScore_P"] = P.numpy()

df_results["BERTScore_R"] = R.numpy()

df_results["BERTScore_F1"] = F1.numpy()


Example 1/10 | mistral @ 0.2
Example 1/10 | mistral @ 0.5
Example 1/10 | mistral @ 0.8
Example 1/10 | llama3 @ 0.2
Example 1/10 | llama3 @ 0.5
Example 1/10 | llama3 @ 0.8
Example 1/10 | gemma2 @ 0.2
Example 1/10 | gemma2 @ 0.5
Example 1/10 | gemma2 @ 0.8
Example 2/10 | mistral @ 0.2
Example 2/10 | mistral @ 0.5
Example 2/10 | mistral @ 0.8
Example 2/10 | llama3 @ 0.2
Example 2/10 | llama3 @ 0.5
Example 2/10 | llama3 @ 0.8
Example 2/10 | gemma2 @ 0.2
Example 2/10 | gemma2 @ 0.5
Example 2/10 | gemma2 @ 0.8
Example 3/10 | mistral @ 0.2
Example 3/10 | mistral @ 0.5
Example 3/10 | mistral @ 0.8
Example 3/10 | llama3 @ 0.2
Example 3/10 | llama3 @ 0.5
Example 3/10 | llama3 @ 0.8
Example 3/10 | gemma2 @ 0.2
Example 3/10 | gemma2 @ 0.5
Example 3/10 | gemma2 @ 0.8
Example 4/10 | mistral @ 0.2
Example 4/10 | mistral @ 0.5
Example 4/10 | mistral @ 0.8
Example 4/10 | llama3 @ 0.2
Example 4/10 | llama3 @ 0.5
Example 4/10 | llama3 @ 0.8
Example 4/10 | gemma2 @ 0.2
Example 4/10 | gemma2 @ 0.5
Example 

c:\Users\joachim.querule\bureau\nlp_ning\.venv-1\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joachim.querule\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 8278.50it/s]
[transformers] RobertaModel 

calculating scores...
computing bert embedding.


100%|██████████| 2/2 [00:03<00:00,  1.75s/it]


computing greedy matching.


100%|██████████| 2/2 [00:00<00:00, 195.53it/s]


done in 3.52 seconds, 25.57 sentences/sec


In [29]:
# 6. SUMMARY

summary = (

    df_results

    .groupby(

        ["model", "temperature"]

    )

    .agg({

        "BLEU": ["mean", "std"],

        "ROUGE-1": ["mean", "std"],

        "ROUGE-L": ["mean", "std"],

        "BERTScore_F1": ["mean", "std"]

    })

    .round(4)

)

print(summary)

                        BLEU          ROUGE-1         ROUGE-L          \
                        mean      std    mean     std    mean     std   
model   temperature                                                     
gemma2  0.2          14.1532  14.1714  0.3753  0.2610  0.3753  0.2610   
        0.5          11.5697   5.8053  0.3984  0.2363  0.3650  0.2283   
        0.8          10.4366   5.8629  0.3269  0.2663  0.2933  0.2573   
llama3  0.2          15.7403   9.9100  0.4744  0.1995  0.4416  0.2192   
        0.5          22.4563  20.4650  0.5277  0.2629  0.4908  0.2941   
        0.8          12.4435   7.8620  0.4240  0.1489  0.4135  0.1528   
mistral 0.2           5.9800   3.9370  0.3003  0.2126  0.2178  0.1484   
        0.5           5.5242   4.1369  0.2777  0.2024  0.2059  0.1273   
        0.8           6.3878   3.8347  0.2741  0.1990  0.2297  0.1615   

                    BERTScore_F1          
                            mean     std  
model   temperature                  

In [30]:

# 7. OPTIONAL: RANK MODELS

ranking = (

    df_results

    .groupby(

        ["model", "temperature"]

    )

    ["BERTScore_F1"]

    .mean()

    .sort_values(

        ascending=False

    )

)

print("\nRanking by BERTScore")

print(ranking)


Ranking by BERTScore
model    temperature
llama3   0.5            0.938715
gemma2   0.5            0.937822
         0.2            0.936838
llama3   0.2            0.934716
gemma2   0.8            0.932081
llama3   0.8            0.928048
mistral  0.5            0.927691
         0.2            0.921344
         0.8            0.919754
Name: BERTScore_F1, dtype: float32


In [31]:
for model_name in models:
    for temp in temperatures:
        print(f"\n=== Examples from model: {model_name} @ temp {temp} ===")
        subset = df_results[(df_results.model == model_name) & (df_results.temperature == temp)].sample(3)
        for _, row in subset.iterrows():
            print(f"Original:  {row['original']}")
            print(f"Reference: {row['reference']}")
            print(f"Paraphrase:{row['paraphrase']}")
            print(f"BLEU: {row['BLEU']:.2f}, ROUGE-1: {row['ROUGE-1']:.2f}, ROUGE-L: {row['ROUGE-L']:.2f}, BERTScore_F1: {row['BERTScore_F1']:.2f}\n")


=== Examples from model: mistral @ temp 0.2 ===
Original:  It’s not true that the boy thinks his team will win.
Reference: The boy doesn’t think his team will win.
Paraphrase:The boy does not believe that his team is going to triumph.
BLEU: 9.24, ROUGE-1: 0.38, ROUGE-L: 0.38, BERTScore_F1: 0.96

Original:  The boy doesn’t know whether the girl came.
Reference: The boy doesn’t know if the girl came.
Paraphrase:The girl's arrival is uncertain to the boy.
BLEU: 6.74, ROUGE-1: 0.44, ROUGE-L: 0.22, BERTScore_F1: 0.91

Original:  The boy doesn’t know that the girl came.
Reference: The boy doesn’t know the girl came.
Paraphrase:The girl's arrival wasn't recognized by the boy.
BLEU: 6.74, ROUGE-1: 0.56, ROUGE-L: 0.33, BERTScore_F1: 0.93


=== Examples from model: mistral @ temp 0.5 ===
Original:  It’s not true that the boy thinks his team will win.
Reference: The boy doesn’t think his team will win.
Paraphrase:The boy does not believe his team is going to be victorious.
BLEU: 9.24, ROUGE-1: 0

# Evaluation & Interpretation

Automatic quality metrics are divided into string-based metrics and machine learning-based metrics.

String-based metrics generally measure the word or character distance between the target sentence and the reference translation.
Examples: BLEU, ROUGE, METEOR, etc

Machine learning-based metrics use sentence embeddings to calculate the difference between the generated target sentence and the reference translation, or even between the target sentence and the source sentence.
Examples: COMET, YiSi, BERTscore

In this tutorial, a good paraphrase should satisfy two objectives simultaneously:

Paraphrase Quality=Semantic Similarity+Lexical Diversity

| Model     | BLEU           | BERTScore                          | 
| --------- | -------------- | ---------------------------------- | 
| High BLEU | High BERTScore | Safe paraphrasing                  |                
| Low BLEU  | High BERTScore | Excellent paraphrasing             |                
| High BLEU | Low BERTScore  | Copying words but altering meaning |                
| Low BLEU  | Low BERTScore  | Poor paraphrasing                  |                


**BLEU, ROUGE, BERTScore**
---

**BLEU** (Bilingual Evaluation Understudy) and **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) are both automatic metrics for evaluating generated text by comparing it against one or more human-written reference texts, but they emphasize different things and are used for different tasks.

**BLEU** was designed for machine translation. It measures **precision**: of the n-grams (word sequences) the generated text produced, how many also appear in the reference? It checks overlap at multiple n-gram lengths (commonly 1 through 4 words) and combines them, with a penalty if the generated text is too short (to discourage gaming the score by outputting only a few "safe" words). The intuition: a good translation should mostly say things that match what a human translator said.

**ROUGE** was designed for summarization. It measures **recall**: of the n-grams (or longer subsequences) in the reference text, how many did the generated text manage to capture? Common variants include ROUGE-N (n-gram overlap), ROUGE-L (longest common subsequence, capturing word order loosely), and ROUGE-S (skip-bigram overlap). The intuition: a good summary should cover the key content that's actually in the reference, even if phrased a bit differently.

| | BLEU | ROUGE |
|---|---|---|
| Primary use | Machine translation | Summarization |
| Core focus | Precision (generated text matches reference) | Recall (reference content is captured) |
| Typical unit | n-grams (1-4 words), with brevity penalty | n-grams, longest common subsequence, skip-bigrams |

Both share the same core limitation: they rely on exact surface-level word/n-gram matching against reference text, so they penalize valid paraphrases and synonyms that don't share exact wording, even when meaning is preserved. This is one motivation for embedding-based evaluation metrics (e.g., BERTScore), which compare meaning via dense vectors rather than exact word overlap — a direct connection back to the word embedding methods we covered earlier, now applied to *evaluating* generated text rather than representing input text.

**BLEU Score Benchmarks**
| BLEU Score | Interpretation                                |
| ---------- | --------------------------------------------- |
| **90–100** | Almost identical (e.g., trivial rewording)    |
| **70–89**  | Very close; high-quality paraphrase or MT     |
| **50–69**  | Good overlap, some variation (common range)   |
| **30–49**  | Acceptable — more syntactic or lexical shifts |
| **<30**    | Very different — maybe creative, maybe wrong  |

In machine translation or paraphrasing:
BLEU ≥ 60 is often very strong
BLEU ≥ 40 is decent
BLEU < 30 may still be fine for very novel or abstract outputs


**ROUGE Score Benchmarks**
| ROUGE Score (F1) | Interpretation                        |
| ---------------- | ------------------------------------- |
| **0.9 – 1.0**    | Almost exact overlap                  |
| **0.7 – 0.89**   | Strong match — captures most info     |
| **0.5 – 0.69**   | Moderate similarity                   |
| **0.3 – 0.49**   | Partial recall — diverges in content  |
| **<0.3**         | Weak — structure/meaning is different |



**BERTScore**

BERTScore is often confusing because there is no official "excellent/good/bad" range like an exam grade.

The score depends on:

the language
the pretrained model used (roberta-large, deberta, etc.)
the dataset
the task itself

For English paraphrase tasks, people usually interpret the F1 score (the main metric) using empirical ranges.

| BERTScore F1 | Interpretation                    | Quality                          |
| ------------ | --------------------------------- | -------------------------------- |
| ≥ 0.95       | Nearly identical sentences        | Excellent / possibly too similar |
| 0.92 – 0.95  | Very strong semantic preservation | Excellent                        |
| 0.88 – 0.92  | Good paraphrase                   | Good                             |
| 0.84 – 0.88  | Acceptable paraphrase             | Moderate                         |
| 0.80 – 0.84  | Weak semantic preservation        | Poor                             |
| < 0.80       | Meaning likely changed            | Unsatisfactory                   |


Read more about ROUGE, BLEAU and other metrics

https://www.digitalocean.com/community/tutorials/automated-metrics-for-evaluating-generated-text

https://clementbm.github.io/theory/2021/12/23/rouge-bleu-scores.html